# Flow Matching Testing
Comprehensive testing for flow matching genre transformation

In [1]:
import torch
from models.flow import FlowMatching
from models.dit import DiT


film_conditioner.py STARTED


In [ ]:

# Fake data
B = 2
C = 1
H = 80
W = 256

x0 = torch.randn(B, C, H, W)  # non-rock
x1 = torch.randn(B, C, H, W)  # rock
genre_ids = torch.tensor([1, 1])  # target = rock

dit = DiT()
flow = FlowMatching(dit)

loss = flow.compute_loss(x0, x1, genre_ids)
print("Loss:", loss.item())


In [ ]:
with torch.no_grad():
    t = torch.rand(B)
    xt = (1 - t.view(B,1,1,1)) * x0 + t.view(B,1,1,1) * x1
    v_true = x1 - x0
    v_pred = dit(xt, t, genre_ids)

print("True velocity norm:", v_true.norm().item())
print("Pred velocity norm:", v_pred.norm().item())


In [ ]:
optimizer = torch.optim.Adam(dit.parameters(), lr=1e-4)

for step in range(200):
    optimizer.zero_grad()
    loss = flow.compute_loss(x0, x1, genre_ids)
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        
        print(f"{step:4d} → {loss.item()}")


In [ ]:
with torch.no_grad():
    x_gen = flow.sample_euler(x0, genre_ids=1, num_steps=50)

dist_start = torch.mean(torch.abs(x0 - x1))
dist_end = torch.mean(torch.abs(x_gen - x1))

print("Distance before:", dist_start.item())
print("Distance after :", dist_end.item())


In [ ]:
with torch.no_grad():
    x_euler = flow.sample_euler(x0, 1, num_steps=30)
    x_heun = flow.sample_heun(x0, 1, num_steps=30)

print("Euler vs Heun difference:",
      torch.mean(torch.abs(x_euler - x_heun)).item())


In [4]:
import torch

mel_tensor = torch.load('../data/output/non_rock_mel/00007_mel.pt')
print(f"Shape: {mel_tensor['mel'].shape}")

Shape: torch.Size([1, 100, 2811])


In [5]:
import torch
from models.dit import DiT
from omegaconf import OmegaConf

# Load config
config = OmegaConf.load('../configs/dit.yaml')

# Initialize model with config
model = DiT(
    patch_height=config.dit_model.patch_height,
    patch_width=config.dit_model.patch_width,
    embed_dim=config.dit_model.embed_dim,
    num_blocks=config.dit_model.num_blocks,
    num_heads=config.dit_model.num_heads,
    num_genres=config.dit_model.num_genres,
    hidden_dim=config.dit_model.hidden_dim,
    dropout=config.dit_model.dropout,
    use_mel_patches=True,
    in_channels=1
)

# Load your actual mel data
mel_tensor = torch.load('../data/output/non_rock_mel/00001_mel.pt')
mel = mel_tensor['mel']  # (1, 100, 2811)

# Add batch dimension
mel = mel.unsqueeze(0)  # (1, 1, 100, 2811)

# Pad width to be divisible by patch_width (64)
pad_width = (64 - (mel.shape[-1] % 64)) % 64
if pad_width > 0:
    mel = torch.nn.functional.pad(mel, (0, pad_width))  # (1, 1, 100, 2816)

print(f"Input shape: {mel.shape}")

# Test forward pass
t = torch.rand(1)  # Random timestep
genre_ids = torch.tensor([1])  # Rock genre

output = model(mel, t, genre_ids)
print(f"Output shape: {output.shape}")
print(f"✅ Dimension test passed!" if output.shape == mel.shape else "❌ Shape mismatch")

Input shape: torch.Size([1, 1, 100, 2816])
Output shape: torch.Size([1, 1, 100, 2816])
✅ Dimension test passed!
